# Filamentation 1030 nm / 13 µJ -- expérience Tokyo

Ce notebook utilise **le paquet `sim/` du dépôt** (`filament_sim.run`,
`figures_filament`, `figures_article`), pas le script monolithique
`NewSim3juillet.py`. Trois raisons concrètes, pas cosmétiques :

1. **La géométrie de cette expérience est déjà codée dans `Config`.**
   Le champ `z_focus_air_um` calcule `begin = -n0 · z_focus_air_um`, et son
   commentaire cite explicitement ton `unified_filament_slider_v3.py`
   (« this is the same rule already used by unified_filament_slider_v3.py »).
   Plus besoin de coder la réfraction à la main.

2. **La convention d'énergie est différente -- et c'est un piège.**
   `NewSim3juillet.py` utilise `I0 = 2E/(π w0² Δt)`, qui traite `Δt` comme une
   durée flat-top : il fallait diviser l'énergie par 1.0645 avant de la passer.
   Le `Config` du dépôt fait **déjà** la correction en interne
   (`P_in = E/(t_p·√(π/2))`, avec le commentaire « overstates the pulse energy
   by 1.0644, i.e. +6.4% »). Donc ici on passe l'énergie **physique vraie**
   (12.561 µJ), *pas* la valeur réduite 11.801. Passer 11.801 sous-estimerait
   l'énergie de 6.4 %.

3. **Le bug `T_op` que j'avais signalé n'existe pas dans ce solveur.**
   `sim/grids.py` a `enable_spectral_filter=True` par défaut, qui masque la
   zone où `T_op < 0` (fréquence absolue négative). Il a en plus
   `enable_space_time_focusing` et les 6 interrupteurs fins de l'éq. (3) du
   rapport, absents de `NewSim3juillet.py`.

**Bonus** : contrairement à `NewSim3juillet.py` (dont `_record()` n'assigne
jamais `E_plasma_z`/`E_MPI_z`), le solveur du dépôt calcule réellement les
pertes d'énergie cumulées -> la figure style Fig. 12 est disponible ici.

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import c as c_SI, epsilon_0, m_e, elementary_charge as q_e

for p in (Path.cwd().parent / "sim", Path.cwd() / "sim"):
    sys.path.insert(0, str(p))

# --- fonctions du depot ---
from filament_sim import run, FIELD_TOGGLES, n_sellmeier
from figures_filament import (
    load_scenario_npz, z_um_of, onaxis_index,
    critical_power, entrance_radius, marburger_collapse, count_refocusing_cycles,
    plot_fig7_fluence_contours, plot_fig8_peak_intensity,
    plot_fig9_electron_density, plot_fig12_energy_losses,
    plot_fig13_free_vs_trapped, plot_free_vs_trapped_vs_z,
    plot_scenario_summary, fluence_level_extent, run_health_check,
)

OUT_ROOT = Path("runs_1030nm"); OUT_ROOT.mkdir(exist_ok=True)
FIG_DIR = OUT_ROOT / "figures_for_report"; FIG_DIR.mkdir(exist_ok=True)
print("FIELD_TOGGLES:", FIELD_TOGGLES)

## 1. Paramètres d'entrée

### Géométrie -- déléguée à `Config.z_focus_air_um`
Le profilomètre mesure le foyer à **272 µm** de la face d'entrée *côté air*.
La réfraction paraxiale à une interface plane conserve le waist `w0` et ne
change que la distance focale (×`n0`). On passe donc `z_focus_air_um=272.0`
et `Config` en déduit `begin = -n0 × 272 µm = -394.4 µm`.

### Énergie -- valeur physique vraie
13.0 µJ incidents → Fresnel (96.63 %) → **12.561 µJ dans le verre**, passés
tels quels à `run()` (voir l'encadré en tête : la correction gaussienne est
déjà dans `Config`).

Contrôle au lancement : `[init] U_beam(0)` doit afficher **≈ 12.56 µJ**.

### `n2` -- le dépôt documente déjà le point
`figures_article.Params` porte le commentaire `n2: 3.54e-20 — défaut de run()
(le notebook ne le surcharge pas)` : le dépôt sait donc déjà que `run()`
impose 3.54e-20 (valeur Couairon 2005 **à 800 nm**) quand on ne dit rien.
Comme l'expérience est à 1030 nm, les trois candidates sont gardées.

| n2 (m²/W) | source | P_cr |
|---|---|---|
| 2.40e-20 | Milam 1998, 800 nm | 4.57 MW |
| **2.74e-20** | Milam 1998, **1053 nm** (retenu) | **4.01 MW** |
| 3.54e-20 | Couairon 2005, 800 nm (défaut `run()`) | 3.10 MW |

La ligne retenue redonne le `P_cr ≈ 4.0 MW` déjà annoncé §8 du rapport.

In [ ]:
# ================= PARAMETRES D'ENTREE =================
# --- geometrie (mesuree cote air ; Config applique la refraction) ---
Z_FOCUS_AIR_UM = 272.0
END_M          = 800e-6

# --- laser ---
WAVELENGTH_M    = 1030e-9
ENERGY_INPUT_UJ = 13.0        # avant l'interface
W0_M            = 3e-6        # waist AU FOYER (inchange par la refraction)
DELTA_T_S       = 263e-15     # FWHM en intensite

TRANSMISSION    = 1.0 - ((1.45 - 1.0) / (1.45 + 1.0))**2      # Fresnel, incidence normale
ENERGY_IN_GLASS = ENERGY_INPUT_UJ * TRANSMISSION              # 12.561 uJ -> passe TEL QUEL

# --- materiau SiO2 (source de chaque valeur en commentaire) ---
N2_MILAM_800  = 2.40e-20   # Milam 1998 @800nm
N2_MILAM_1053 = 2.74e-20   # Milam 1998 @1053nm  <- le plus proche de 1030 nm
N2_COUAIRON   = 3.54e-20   # Couairon 2005 @800nm (defaut de run())
N2_CHOSEN     = N2_MILAM_1053

UI_EV       = 9.0        # gap SiO2         -- Couairon 2005 / Bulgakova 2010
MEFF_REL    = 0.64       # masse Keldysh    -- Couairon 2005
TAU_C_S     = 1.7e-15    # -> omega0*tau_c = 3.1 a 1030 nm (convention Bulgakova ;
                          #    Couairon utilise 1e-14 -> 23.6). Defaut du depot.
TAU_R_S     = 330e-15    # piegeage STE     -- Mouskeftaras 2013 / Tsaturyan 2025
RHO_MAX_CM3 = 2.1e22     # densite d'atomes -- Couairon 2005
US_EV       = 6.0        # re-ionisation STE-- Chimier PRB 2011
F_R, TAU_D_S, TAU_S_S = 0.18, 32e-15, 12e-15   # Raman -- Couairon 2005
LAMBDA_PROBE_M = 490e-9

# --- grille ---
NT = 4096       # cf. section 2 : Nt=2000 de l'ancien notebook etait trop serre
NR = 3001       # ordre Hankel -> 3000 points radiaux
R_FACTOR = 90.0
SAVE_STRIDE, RHO_T_STRIDE, RHO_R_STRIDE = 100, 20, 4

# begin est calcule par Config, mais on le reproduit ici pour dimensionner Nz
n0 = n_sellmeier(WAVELENGTH_M)
BEGIN_M = -n0 * Z_FOCUS_AIR_UM * 1e-6
LZ = END_M - BEGIN_M
NZ = int(LZ / 24e-9)

k0 = 2*np.pi*n0/WAVELENGTH_M
Z_R_FOCAL = k0*W0_M**2/2
W_ENTRANCE_M = entrance_radius(W0_M, BEGIN_M, WAVELENGTH_M, n0)

print(f"n0(1030nm)        = {n0:.5f}   (Sellmeier ; le slider suppose 1.45 -> accord 0.003%)")
print(f"begin             = {BEGIN_M*1e6:+.2f} um   (= -n0 x {Z_FOCUS_AIR_UM:.0f} um, calcule par Config)")
print(f"boite             = [{BEGIN_M*1e6:+.1f}, {END_M*1e6:+.1f}] um   Nz={NZ}  dz={LZ/NZ*1e9:.2f} nm")
print(f"energie           = {ENERGY_INPUT_UJ:.1f} uJ -> {ENERGY_IN_GLASS:.3f} uJ (T={TRANSMISSION*100:.2f}%)")
print(f"                    passee TELLE QUELLE a run() (Config fait deja la correction gaussienne)")
print(f"z_R au foyer      = {Z_R_FOCAL*1e6:.2f} um")
print(f"rayon a l'ENTREE  = {W_ENTRANCE_M*1e6:.2f} um   (x{W_ENTRANCE_M/W0_M:.1f} le waist focal)")

## 2. Vérification de la grille

Reprise de l'audit fait sur l'ancien notebook, avec deux points qui
**disparaissent** en passant au solveur du dépôt.

| | verdict |
|---|---|
| `dz = 24 nm` | OK, voire large : phase Kerr/pas ≈ 4e-3 rad (critère 0.05), 184 pas par z_R d'un cœur de 1 µm. `dz=48 nm` diviserait le coût par 2. |
| `dr ≈ 90 nm` | OK : 33 points dans w₀, 11 dans un cœur de 1 µm. Marges absorbeur ×8.1 (entrée) / ×4.0 (sortie). |
| `Nt = 4096` | Relevé depuis 2000. À 1030 nm l'ordre multiphotonique est **K = 8** (contre 6 à 800 nm), donc l'ionisation est *plus* sensible à l'intensité et demande un pas temporel fin. |
| `T_op < 0` | **Non applicable ici** : `enable_spectral_filter=True` masque déjà la zone de fréquence absolue négative. C'était un défaut de `NewSim3juillet.py` seulement. |
| pertes d'énergie | **Disponibles ici** (`E_MPI_z`/`E_plasma_z` réellement calculés), contrairement à `NewSim3juillet.py`. |
| taille du npz | `rho_t_stride=20` **et** `rho_r_stride=4` (ce dernier n'existe pas dans `NewSim3juillet.py`) → cubes divisés par 8 par rapport à l'ancien réglage. |

In [ ]:
from scipy.special import jn_zeros
dz = LZ/NZ
print("=== LONGITUDINAL ===")
print(f"dz={dz*1e9:.2f} nm  Nz={NZ}   phase Kerr/pas @I_clamp = {k0*N2_CHOSEN*5e17*dz:.2e} rad  (<0.05)")
print(f"z_R(coeur 1um)={k0*1e-12/2*1e6:.2f} um -> {k0*1e-12/2/dz:.0f} pas/z_R")

print("\n=== RADIAL ===")
j = jn_zeros(0, NR); R = R_FACTOR*W0_M
rlist = j[:NR-1]*R/j[NR-1]; dr = np.diff(rlist).mean()
w_end = W0_M*np.sqrt(1+(END_M/Z_R_FOCAL)**2)
print(f"R_max={R*1e6:.1f} um  absorbeur={0.9*R*1e6:.1f} um  dr={dr*1e6:.4f} um")
print(f"  {W0_M/dr:.1f} pts dans w0 | {1e-6/dr:.1f} pts dans 1 um")
print(f"  marges: entree x{0.9*R/W_ENTRANCE_M:.1f} ({W_ENTRANCE_M*1e6:.1f} um), sortie x{0.9*R/w_end:.1f} ({w_end*1e6:.1f} um)")

print("\n=== TEMPOREL ===")
tp = DELTA_T_S/np.sqrt(2*np.log(2)); tmax = 5*tp; dt = 2*tmax/NT
f0 = c_SI/WAVELENGTH_M
E_ph = 1240/(WAVELENGTH_M*1e9)
print(f"tp={tp*1e15:.1f} fs  tmax={tmax*1e15:.0f} fs  dt={dt*1e15:.3f} fs  f_Nyq/f0={1/(2*dt)/f0:.2f}")
print(f"E_photon={E_ph:.3f} eV -> K = ceil({UI_EV}/{E_ph:.3f}) = {int(np.ceil(UI_EV/E_ph))} photons")
print("T_op : protege par enable_spectral_filter=True (grids.py) -- pas de bin negatif")

print("\n=== SAUVEGARDE ===")
n_saves = NZ//SAVE_STRIDE+1
Nt_sub = (NT-1)//RHO_T_STRIDE+1; Nr_sub = (NR-1-1)//RHO_R_STRIDE+1
print(f"n_saves={n_saves} (dz_save={dz*SAVE_STRIDE*1e6:.2f} um) | Nt_sub={Nt_sub} | Nr_sub={Nr_sub}")
print(f"3 cubes (z,r,t) = {3*n_saves*Nr_sub*Nt_sub*4/1e9:.2f} GB   (vs 3.59 GB avec l'ancien reglage)")
print(f"tableaux 2D     = {4*n_saves*(NR-1)*4/1e6:.0f} MB")

## 3. P_cr et L_c -- prédiction testable

`marburger_collapse()` et `entrance_radius()` viennent de `figures_filament`.

Point crucial encodé dans `entrance_radius()` : `L_DF = k w²/2` doit utiliser
le rayon **au plan d'entrée** (29.9 µm ici), pas le waist focal (3 µm). Le
solveur lance `envelope_gaussian_focused` avec `curv = 1 + 2i·begin/b`, donc
le faisceau fait déjà 10× son waist en entrant. Utiliser 3 µm sous-estime
`L_c` d'un facteur ~100 et placerait le collapse collé à la face d'entrée.

In [ ]:
P_in = ENERGY_IN_GLASS*1e-6 / (tp*np.sqrt(np.pi/2))   # meme formule que Config
F_EXT = abs(BEGIN_M)

PRED = {}
for lab, n2 in (("Milam 800nm", N2_MILAM_800),
                ("Milam 1053nm", N2_MILAM_1053),
                ("Couairon", N2_COUAIRON)):
    P_cr = critical_power(n2, WAVELENGTH_M, n0)
    ratio, L_DF, L_c, L_cf = marburger_collapse(P_in, P_cr, W_ENTRANCE_M,
                                                 WAVELENGTH_M, n0, F_EXT)
    PRED[lab] = dict(n2=n2, P_cr=P_cr, ratio=ratio, L_cf=L_cf,
                     z_pred_um=L_cf*1e6 + BEGIN_M*1e6)
    print(f"{lab:13s} n2={n2:.2e}  P_cr={P_cr*1e-6:5.2f} MW  P_in/P_cr={ratio:5.2f}")
    print(f"{'':13s} L_DF={L_DF*1e6:6.0f} um  L_c={L_c*1e6:6.1f} um  L_c,f={L_cf*1e6:6.1f} um"
          f"  -> collapse z_sim={PRED[lab]['z_pred_um']:+7.1f} um")

print(f"\nP_in = {P_in*1e-6:.2f} MW")
print(f"face d'entree z_sim={BEGIN_M*1e6:+.1f} um | foyer geometrique z_sim=0")
print("=> le foyer NON-LINEAIRE est predit ~175 um AVANT le foyer geometrique.")

# lignes verticales reutilisees par toutes les figures
VLINES = [(PRED[l]["z_pred_um"], f"L_c,f {l}", c)
          for l, c in zip(PRED, ("tab:blue", "tab:green", "tab:orange"))]
VLINES += [(0.0, "foyer geometrique", "purple"), (BEGIN_M*1e6, "face d'entree", "green")]

## 4. Lancer (ou recharger)

`load_scenario_npz` (du dépôt) relit un run existant et détecte
automatiquement un cache périmé via l'empreinte du code source.

**Non exécuté ici** (pas de GPU dans cet environnement). Compter plusieurs
heures : ~49 800 pas, chacun avec 4 produits Hankel denses 3000×3000.

In [ ]:
OUT_DIR = str(OUT_ROOT / "filament_13uJ_w3um")

res = load_scenario_npz(OUT_DIR)
if res is None:
    print(f"Lancement -> {OUT_DIR}   (plusieurs heures de GPU)")
    res = run(
        # grille
        Nz=NZ, Nt=NT, Nr=NR, R_factor=R_FACTOR,
        end=END_M,
        z_focus_air_um=Z_FOCUS_AIR_UM,   # -> Config calcule begin = -n0*272um
        save_stride=SAVE_STRIDE, ckpt_every=500, verbose=True,
        # laser
        wavelength=WAVELENGTH_M,
        energy_uJ=ENERGY_IN_GLASS,       # energie VRAIE (Config corrige deja)
        w0=W0_M, delta_t=DELTA_T_S,
        # materiau
        n2=N2_CHOSEN, Ui_eV=UI_EV, meff_rel=MEFF_REL,
        tau_c=TAU_C_S, tau_r=TAU_R_S, rho_max=RHO_MAX_CM3,
        Us_eV=US_EV, f_R=F_R, tau_d=TAU_D_S, tau_s=TAU_S_S,
        enable_ste=True,
        # sonde + enregistrement
        lambda_probe=LAMBDA_PROBE_M,
        rho_t_stride=RHO_T_STRIDE, rho_r_stride=RHO_R_STRIDE,
        out_dir=OUT_DIR, envelope="gaussian_focused",
    )
else:
    print(f"Recharge depuis {OUT_DIR}/result.npz")

print(f"\nControle : [init] U_beam(0) doit valoir ~{ENERGY_IN_GLASS:.2f} uJ")

In [ ]:
run_health_check(res, out_dir=OUT_DIR, label="filament 13uJ 1030nm",
                 rho_max=RHO_MAX_CM3)
fluence_level_extent(res, levels=(1.0, 5.0, 20.0), label="13 uJ")

## 5. Vue d'ensemble 2×2

In [ ]:
NC_PROBE = epsilon_0*m_e*(2*np.pi*c_SI/LAMBDA_PROBE_M)**2/q_e**2*1e-6
plot_scenario_summary(res, "filament_13uJ_1030nm",
                      nc_probe_cm3=NC_PROBE,
                      rho_lines=(1e20, RHO_MAX_CM3),
                      fluence_levels=(1., 5., 20.),
                      save=str(FIG_DIR/"summary_1030nm.png"));

## 6. Fluence -- cycles de focalisation / défocalisation

In [ ]:
plot_fig7_fluence_contours(res, levels=(1., 2., 5., 10., 20.),
                           label="13 µJ, 1030 nm", vlines=VLINES,
                           save=str(FIG_DIR/"fluence_contours_1030nm.png"));

## 7. Intensité crête -- L_c,f prédite vs collapse observé

In [ ]:
fig = plot_fig8_peak_intensity({"13 µJ, 1030 nm": res})
ax = fig.axes[0]
for zv, lab, col in VLINES:
    ax.axvline(zv, ls="--", lw=1.1, color=col, label=lab)
ax.legend(fontsize=7, ncol=2)
fig.savefig(FIG_DIR/"peak_intensity_vs_Lc_1030nm.png", dpi=150)

## 8. Électrons libres et excitons auto-piégés

Deux vues complémentaires : le long de `z` (maximum temporel), puis en
fonction du **temps** au foyer non-linéaire.

In [ ]:
plot_free_vs_trapped_vs_z(res, rho_max_cm3=RHO_MAX_CM3, nc_probe_cm3=NC_PROBE,
                          vlines=VLINES,
                          save=str(FIG_DIR/"rho_e_rho_s_vs_z_1030nm.png"));

In [ ]:
plot_fig13_free_vs_trapped(res, label="13 µJ, 1030 nm",
                           save=str(FIG_DIR/"free_vs_trapped_vs_t_1030nm.png"));

## 9. Pertes d'énergie

Impossible avec `NewSim3juillet.py` (tableaux jamais remplis), disponible ici.

In [ ]:
plot_fig12_energy_losses(res, label="13 µJ, 1030 nm",
                         xlim=(BEGIN_M*1e6, END_M*1e6), ylim=(1e-3, 1e0),
                         save=str(FIG_DIR/"energy_losses_1030nm.png"));

## 10. Comptage des cycles + confrontation à L_c,f

In [ ]:
idx, z_cycles = count_refocusing_cycles(res, I_clamp=5e13)
if len(idx):
    z_obs = z_cycles[0]
    print(f"\nPremier collapse OBSERVE a z_sim = {z_obs:+.1f} um")
    for lab, p in PRED.items():
        print(f"  vs {lab:13s}: predit {p['z_pred_um']:+7.1f} um   ecart {z_obs-p['z_pred_um']:+7.1f} um")

## 11. Figures « articles » du dépôt

`figures_article` réintègre l'équation de population 0D à partir de la trace
d'intensité on-axis du run -- utile pour vérifier que le noyau CUDA et le
modèle analytique disent la même chose.

In [ ]:
from figures_article import Params as ArticleParams, fig2_populations, fig13_electron_density_vs_z

# Memes constantes que le run, sinon la reintegration 0D diverge du noyau CUDA
# (tau_c en particulier : defaut de Params = 1.7e-15, a re-passer explicitement
# si BASE change un jour).
prm = ArticleParams(
    wavelength=WAVELENGTH_M, Ui_eV=UI_EV, Us_eV=US_EV, meff_rel=MEFF_REL,
    tau_c=TAU_C_S, tau_r=TAU_R_S, rho_max=RHO_MAX_CM3,
    n2=N2_CHOSEN, lambda_probe=LAMBDA_PROBE_M,
)
fig2_populations(res, prm, save=str(FIG_DIR/"fig2_populations_1030nm.png"))
fig13_electron_density_vs_z(res, prm, save=str(FIG_DIR/"fig13_density_vs_z_1030nm.png"));

## 12. Récapitulatif pour `main.tex`

In [ ]:
print("Figures ->", FIG_DIR.resolve())
for f in sorted(FIG_DIR.glob("*.png")): print("  -", f.name)

z_um = z_um_of(res); Imax = np.asarray(res["Imax_z"]); ia = onaxis_index(res)
print(f"\n1030 nm | {ENERGY_INPUT_UJ} uJ incident -> {ENERGY_IN_GLASS:.2f} uJ dans le verre")
print(f"w0(foyer)={W0_M*1e6:.1f} um | w(entree)={W_ENTRANCE_M*1e6:.1f} um | FWHM={DELTA_T_S*1e15:.0f} fs")
print(f"n2={N2_CHOSEN:.2e} m2/W -> P_cr={PRED['Milam 1053nm']['P_cr']*1e-6:.2f} MW, "
      f"P_in/P_cr={PRED['Milam 1053nm']['ratio']:.1f}")
print(f"I_max={Imax.max():.3e} W/cm2 @ z={z_um[np.argmax(Imax)]:+.1f} um")
print(f"rho_e max={res['rho_rz'][:,ia].max():.3e} cm-3 | rho_s max={res['rho_s_rz'][:,ia].max():.3e} cm-3")
print(f"pertes totales={np.max(res['E_total_z'])*100:.1f} %")
print(f"{len(idx)} cycle(s) detecte(s)")